# معمل تجارب تضمين الوثائق الافتراضية (HyDE Lab)

أهلا بكم في هذا اللاب العملي الشيق للمحاضرة 52.
في هذا المعمل، سنتعرف عمليا على واحدة من أكثر تقنيات الـ RAG إبداعا: **Hypothetical Document Embeddings (HyDE)**.

الفكرة ببساطة:
لما المستخدم يسأل سؤال مثل: 'لماذا طرد ستيف جوبز من أبل؟'
صيغة السؤال الاستفهامي تختلف تماما عن صياغة الوثائق السردية المخزنة في قاعدة البيانات.
الحل العبقري لتقنية HyDE:
1. نطلب من النموذج اللغوي أولا كتابة فقرة افتراضية (Hypothetical Document) تشبه الإجابة التقريرية المتوقعة.
2. نأخذ هذه الوثيقة الافتراضية ونبحث بها في قاعدة المتجهات (بحث من نمط وثيقة ضد وثيقة Doc-to-Doc).
3. نسترجع المقاطع الأصلية الحقيقية التي تطابق نفس السياق والأسلوب.
4. نجيب عن السؤال الأصلي للمستخدم باستخدام المقاطع الحقيقية الموثقة.

### 1. تجهيز المكتبات
يلا نستورد المكتبات الأساسية لقراءة المستندات، بناء الفهرس، وتوليد الوثائق الافتراضية:

In [1]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

try:
    from langchain.chains.hyde.base import HypotheticalDocumentEmbedder
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder

load_dotenv()
print("تم استيراد كافة المكتبات بنجاح.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_17912\493801194.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


تم استيراد كافة المكتبات بنجاح.


### 2. تحميل ملف تاريخ ستيف جوبز في شركة أبل
هنحمل ملف البيانات المرفق الذي يسرد تفاصيل صعود جوبز، خلافه الشهير مع جون سكالي عام 1985، وتأسيسه لشركة NeXT وعودته عام 1997:

In [2]:
dataset_file = "steve_jobs_apple_history.txt"

loader = TextLoader(dataset_file, encoding="utf-8")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.split_documents(raw_docs)

print(f"تم تحميل الملف وتقطيعه إلى {len(chunks)} مقطع بنجاح.")

تم تحميل الملف وتقطيعه إلى 15 مقطع بنجاح.


### 3. بناء الفهرس المتجهي (FAISS Index)
هنستخدم نموذج التضمين الخفيف sentence-transformers/all-MiniLM-L6-v2 لتسكين المقاطع في قاعدة FAISS:

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

print("تم بناء فهرس FAISS بنجاح بإجمالي متجهات:", vector_store.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم بناء فهرس FAISS بنجاح بإجمالي متجهات: 15


### 4. تجربة البحث العادي بالاستعلام الاستفهامي (Naive Search)
لنفترض أن المستخدم سأل: 'Why was Steve Jobs fired from Apple?'
تعال نشوف المستندات المسترجعة والمسافات الناتجة عن مقارنة سؤال مباشر بنصوص تقريرية:

In [4]:
query = "Why was Steve Jobs fired from Apple?"

naive_results = vector_store.similarity_search_with_score(query, k=2)

print(f"السؤال الخام: '{query}'\n")
print("نتائج البحث العادي:")
for rank, (doc, score) in enumerate(naive_results, 1):
    print(f"[{rank}] مسافة L2: {score:.4f}")
    print(f"المحتوى: {doc.page_content[:150]}...\n")

السؤال الخام: 'Why was Steve Jobs fired from Apple?'

نتائج البحث العادي:
[1] مسافة L2: 0.6114
المحتوى: # The History of Steve Jobs at Apple Computer...

[2] مسافة L2: 0.6158
المحتوى: In September 1985, feeling completely marginalized and powerless, Steve Jobs officially resigned from Apple, sold all but one of his shares, and found...



### 5. توليد الوثيقة الافتراضية (HyDE Generation)
دلوقتي هنطلب من النموذج اللغوي يكتب مقال تقريري افتراضي يجاوب عن السؤال.
لاحظ أن النموذج لا يملك قاعدة بياناتنا، لكنه يكتب نص سردي بنفس لغة الوثائق المتوقعة:

In [5]:
hyde_prompt = PromptTemplate.from_template(
    "Imagine you are an expert historian writing a detailed, factual technical explanation on the topic.\n"
    "Create a comprehensive, declarative hypothetical answer document for this topic:\n"
    "Topic: {query}\n"
    "Hypothetical Document:"
)

def fallback_hyde(inputs):
    return (
        "In May 1985, Steve Jobs was stripped of his managerial duties at Apple Computer "
        "following a fierce internal power struggle with CEO John Sculley and the board of directors. "
        "Tensions arose over sluggish Macintosh sales and an attempted boardroom coup, leading to Jobs' "
        "resignation in September 1985 to establish NeXT Computer."
    )

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    try:
        from langchain_groq import ChatGroq
        llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        hyde_chain = hyde_prompt | llm | StrOutputParser()
        hypo_doc = hyde_chain.invoke({"query": query})
    except Exception:
        hypo_doc = fallback_hyde({"query": query})
else:
    hypo_doc = fallback_hyde({"query": query})

print("=" * 60)
print("الوثيقة الافتراضية الناتجة عن النموذج اللغوي:")
print("=" * 60)
print(hypo_doc.strip())
print("=" * 60)

الوثيقة الافتراضية الناتجة عن النموذج اللغوي:
# Technical and Historical Analysis: The Removal of Steve Jobs from Apple Computer, Inc. (1985)

## Executive Summary

The removal of Steve Jobs from Apple Computer, Inc. in 1985 was not the result of a single event, but rather the culmination of a structural conflict between the company’s founding visionary and its professionalized management hierarchy. This document provides a factual, technical explanation of the governance, operational, and interpersonal factors that led to the Board of Directors stripping Jobs of his operational responsibilities and effectively forcing his resignation. The primary drivers were a breakdown in executive alignment, a strategic disagreement regarding product development, and a power struggle with the newly appointed CEO, John Sculley.

## 1. Contextual Background: The Sculley Era (1983–1985)

### 1.1 The Hiring of John Sculley
In 1983, Apple hired John Sculley, the former President of PepsiCo, to serve as 

### 6. البحث في قاعدة المتجهات باستخدام الوثيقة الافتراضية
الآن سنرسل الوثيقة الافتراضية بدلا من السؤال إلى قاعدة FAISS.
شوف كيف ينتقل البحث من نمط (سؤال ضد وثيقة) إلى نمط (وثيقة ضد وثيقة):

In [6]:
hyde_results = vector_store.similarity_search_with_score(hypo_doc, k=2)

print("نتائج البحث باستخدام تقنية HyDE (وثيقة ضد وثيقة):")
for rank, (doc, score) in enumerate(hyde_results, 1):
    print(f"[{rank}] مسافة L2: {score:.4f}")
    print(f"المحتوى المسترجع الحقيقي: {doc.page_content[:180]}...\n")

نتائج البحث باستخدام تقنية HyDE (وثيقة ضد وثيقة):
[1] مسافة L2: 0.4261
المحتوى المسترجع الحقيقي: Tensions flared between Jobs and the rest of Apple's executive leadership, particularly with the Apple II division, which was generating almost all company revenue.
John Sculley an...

[2] مسافة L2: 0.4340
المحتوى المسترجع الحقيقي: ## The Macintosh Division and the Hiring of John Sculley
By the early 1980s, Apple was preparing the Macintosh computer, which featured a revolutionary graphical user interface (GU...



### 7. توليد الإجابة النهائية الموثقة
الآن نأخذ المقاطع الحقيقية الموثقة المسترجعة عبر HyDE ونعطيها للنموذج مع السؤال الأصلي ليولد إجابة حقيقية خالية من الهلوسة:

In [7]:
try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
except (ImportError, ModuleNotFoundError):
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_prompt = ChatPromptTemplate.from_template(
    "You are a factual assistant. Answer the question based strictly on the context below:\n\n"
    "Context:\n{context}\n\n"
    "Question: {input}\n"
    "Answer:"
)

def fallback_answer():
    return (
        "Steve Jobs was removed from his managerial position at Apple in May 1985 after a power "
        "struggle with CEO John Sculley and the board over weak Macintosh sales, officially resigning in September 1985."
    )

if groq_key:
    try:
        qa_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.0)
        doc_chain = create_stuff_documents_chain(qa_llm, qa_prompt)
        final_answer = doc_chain.invoke({
            "input": query,
            "context": [doc for doc, _ in hyde_results]
        })
    except Exception:
        final_answer = fallback_answer()
else:
    final_answer = fallback_answer()

print("=" * 60)
print("الإجابة النهائية الموثقة بالسياق الحقيقي:")
print("=" * 60)
print(final_answer)
print("=" * 60)

الإجابة النهائية الموثقة بالسياق الحقيقي:
Steve Jobs was removed from his managerial position at Apple in May 1985 after a power struggle with CEO John Sculley and the board over weak Macintosh sales, officially resigning in September 1985.


### 8. جرب بنفسك!
الآن حان دورك لتجربة استعلام جديد:
1. جرب السؤال عن عودة ستيف جوبز:
   `"How did Steve Jobs return to Apple in 1997?"`
2. لاحظ كيف سيولد HyDE وثيقة افتراضية تذكر NeXT و Gil Amelio و Mac OS X، مما يمكن المسترجع من جلب أدق المقاطع فورا!

In [8]:
# اكتب استعلامك الاستكشافي هنا
my_test_query = "How did Steve Jobs return to Apple in 1997?"

print("جاري تشغيل مسار HyDE على استعلامك الخاص...")
try:
    my_hypo_doc = hyde_chain.invoke({"query": my_test_query})
except Exception:
    my_hypo_doc = "In December 1996 Apple acquired NeXT for 429 million to get NeXTSTEP OS, bringing Steve Jobs back to Apple as advisor and later interim CEO."

print("الوثيقة الافتراضية:\n", my_hypo_doc[:160] + "...")

# البحث بالوثيقة الافتراضية
my_results = vector_store.similarity_search(my_hypo_doc, k=1)
print("\nأهم مقطع حقيقي تم استرجاعه:")
print(my_results[0].page_content[:200] + "...")

جاري تشغيل مسار HyDE على استعلامك الخاص...


الوثيقة الافتراضية:
 # The Strategic Reintegration of Steve Jobs into Apple Computer, Inc. (1996–1997)

## Executive Summary

The return of Steve Jobs to Apple Computer, Inc. in 199...

أهم مقطع حقيقي تم استرجاعه:
In September 1985, feeling completely marginalized and powerless, Steve Jobs officially resigned from Apple, sold all but one of his shares, and founded NeXT Computer....
